# 03 — Bigram language model

## Recap and `get_batch`
Same data pipeline as notebook 01: character tokenizer → 1-D tensor → 80/20 split →
random `(batch_size, block_size)` chunks of inputs `x` and shifted targets `y`.

In [1]:
import torch
import torch.nn as nn
from torch.nn import functional as F

device = 'cuda' if torch.cuda.is_available() else ('mps' if torch.backends.mps.is_available() else 'cpu')

# hyperparameters
block_size = 8
batch_size = 32
max_iters = 10000
eval_interval = 1000
learning_rate = 1e-2  # a tiny model tolerates a large lr; the GPT uses 3e-4
eval_iters = 250
torch.manual_seed(1337)

with open('../data/wizard_of_oz.txt', 'r', encoding='utf-8') as f:
    text = f.read()
chars = sorted(set(text))
vocab_size = len(chars)
string_to_int = {ch: i for i, ch in enumerate(chars)}
int_to_string = {i: ch for i, ch in enumerate(chars)}
encode = lambda s: [string_to_int[c] for c in s]
decode = lambda l: ''.join(int_to_string[i] for i in l)

data = torch.tensor(encode(text), dtype=torch.long)
n = int(0.8 * len(data))
train_data, val_data = data[:n], data[n:]

def get_batch(split):
    data = train_data if split == 'train' else val_data
    ix = torch.randint(len(data) - block_size, (batch_size,))
    x = torch.stack([data[i:i + block_size] for i in ix])
    y = torch.stack([data[i + 1:i + block_size + 1] for i in ix])
    return x.to(device), y.to(device)

## Loss reporting + train vs eval mode

Loss of a single batch is noisy, so we average over `eval_iters` batches.
`@torch.no_grad()` skips building the autograd graph (faster, less memory) and
`model.eval()` switches layers such as dropout to inference behaviour; `model.train()`
switches back.

In [2]:
@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train', 'val']:
        losses = torch.zeros(eval_iters)
        for k in range(eval_iters):
            X, Y = get_batch(split)
            logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

## `nn.Module` subclass, logits and reshaping

The whole bigram model is a single `vocab_size × vocab_size` embedding table: row *i*
holds the **logits** (unnormalised scores) for the character that follows character *i*.

* `forward` returns logits of shape `(B, T, V)`.
* `F.cross_entropy` wants `(N, C)` so we reshape to `(B*T, V)` and targets to `(B*T,)`.
  Cross-entropy = `-log softmax(logits)[target]`; a random model scores ≈ `ln(V)`.

## Generate function and giving the model some context

`generate` repeatedly: runs the model, takes logits of the **last** time step
(`logits[:, -1, :]`, shape `(B, V)` — logits dimensionality), turns them into
probabilities with softmax, samples one token with `torch.multinomial`, and appends it
to the context with `torch.cat`.

In [3]:
class BigramLanguageModel(nn.Module):
    def __init__(self, vocab_size):
        super().__init__()
        self.token_embedding_table = nn.Embedding(vocab_size, vocab_size)

    def forward(self, index, targets=None):
        logits = self.token_embedding_table(index)          # (B, T, V)
        if targets is None:
            loss = None
        else:
            B, T, C = logits.shape
            logits = logits.view(B * T, C)
            targets = targets.view(B * T)
            loss = F.cross_entropy(logits, targets)
        return logits, loss

    def generate(self, index, max_new_tokens):
        # index is (B, T) array of indices in the current context
        for _ in range(max_new_tokens):
            logits, loss = self.forward(index)
            logits = logits[:, -1, :]                        # (B, V) last time step
            probs = F.softmax(logits, dim=-1)                # (B, V)
            index_next = torch.multinomial(probs, num_samples=1)  # (B, 1)
            index = torch.cat((index, index_next), dim=1)    # (B, T+1)
        return index

model = BigramLanguageModel(vocab_size).to(device)

context = torch.zeros((1, 1), dtype=torch.long, device=device)
print('untrained:\n', decode(model.generate(context, max_new_tokens=300)[0].tolist()))

untrained:
 
eoM8MIlMhxi)PS1;.i6&M8Ai:WLC7ROip90wQUB.aF8_!?wEK:4)u7RZPPSV,J:D&E3sKYC.jxm9a?F?FloThiSc0NC4afnZ]S.Ti;pPBPGCs"trZ
vpZRYEBO*RYCL kJ[!xra_dQFeE7uMh_w5O"9V;uN)KV)X-yFWsX7sDX01fmvMp E,0vnR71*bvto,puIYik *bAbyar-B13me-z'91IY6B'8VGC(3rj4_c?2hKj9;*7toj5)RIYP&gLzRI*AS]i6&"CTK.Gzl8eAY[e]?'8u;dh15Fn8-KIq!AtLg


## Gradient descent

Loss is a function of every parameter. The gradient `∂loss/∂θ` points uphill, so we
take a small step the other way: `θ ← θ − lr · ∂loss/∂θ`. `loss.backward()` computes
all gradients via autograd (back-propagation / chain rule).

## Training loop + optimizer + `zero_grad`

1. sample a batch, 2. forward → loss, 3. `optimizer.zero_grad(set_to_none=True)` —
PyTorch **accumulates** gradients, so clear the previous step's, 4. `loss.backward()`,
5. `optimizer.step()` updates parameters.

## Optimizers overview

* **SGD** — `θ -= lr·g`. Simple, noisy.
* **Momentum** — keeps a running velocity of past gradients to smooth the path.
* **RMSprop** — divides by a running RMS of gradients → per-parameter step size.
* **Adam** — momentum + RMSprop with bias correction; the default for deep nets.
* **AdamW** — Adam with *decoupled* weight decay (regularisation); standard for
  transformers and what we use.

See `docs/03_training_and_optimizers.md` for where each is applied.

In [4]:
optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)

for iter in range(max_iters):
    if iter % eval_interval == 0:
        losses = estimate_loss()
        print(f"step: {iter}, train loss: {losses['train']:.3f}, val loss: {losses['val']:.3f}")

    xb, yb = get_batch('train')
    logits, loss = model.forward(xb, yb)
    optimizer.zero_grad(set_to_none=True)
    loss.backward()
    optimizer.step()

print(loss.item())

step: 0, train loss: 4.826, val loss: 4.830


step: 1000, train loss: 2.468, val loss: 2.512


step: 2000, train loss: 2.429, val loss: 2.486


step: 3000, train loss: 2.443, val loss: 2.482


step: 4000, train loss: 2.428, val loss: 2.479


...

step: 6000, train loss: 2.425, val loss: 2.477


step: 7000, train loss: 2.424, val loss: 2.470


step: 8000, train loss: 2.423, val loss: 2.466


step: 9000, train loss: 2.424, val loss: 2.472


2.4376113414764404


In [5]:
context = torch.zeros((1, 1), dtype=torch.long, device=device)
print(decode(model.generate(context, max_new_tokens=500)[0].tolist()))


wadisumm?" oung maney_ ank s o theialooug
a,"It d ce he ty ghime re te
"Ozagl.
ckstoreaweclur t scarid nto fithean whe he y.
y poaled win d wir I fofarrod tund we anogeepem o s bolofondlleraigesery ansh s he an ustil tronding we s antheathesid u de acithabomowad  d, ie sthemno s dispan, ss pites hthallicang angrs t!"OF fure at and d Vam ttarnonled ts o wauarous he t inof a he yosoure.


CI's, d byosomedereven CHAnd as ayo Widom.
"Wid ooft tthor o stindirherizale teroorknomin ageano n alllid ut w


The output now has plausible letter pairs, spaces and punctuation, but no words — a
bigram only ever sees one character of context. To do better, tokens must *talk to
each other*: that is what self-attention does (notebooks 05–06).